# NB19: External validation forest plot (Figure 4A)

Aggregates external validation results from NB14, NB15, NB16, NB17 into
a single forest plot of AUROCs with 95% bootstrap CIs across all
external cohorts and a consolidated CSV summary table.

**Manuscript reference** (Figure 4A, Tables 2 + 3):
- CPTAC-LUAD (frozen TCGA OpenCLIP, NB14): AUROC = 0.671 (0.487-0.825)
- CPTAC-LUAD (OSFM within-cohort,    NB15): AUROC = 0.723
- CPTAC-LUSC (OSFM within-cohort,    NB15): AUROC = 0.527
- CPTAC-HNSCC (OSFM within-cohort,   NB15): AUROC = 0.475
- PTRC-HGSOC platinum resistance     (NB16): AUROC = 0.673 (0.588-0.757)
- SurGen-CRC MMR off-target (frozen, NB17): AUROC = 0.674 (0.55-0.79)
- TCGA-UCEC MMR off-target (frozen,  NB17): AUROC = 0.445 (0.353-0.545)

**Required env**: `WORKSPACE`. **Inputs**: per-cohort metric files written
by NB14-NB17.
**Outputs**: `figures/fig4a_forest_external.{png,pdf}`,
`results/external_consolidated/external_summary.csv`.

In [ ]:
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

WORKSPACE = Path(os.environ.get("WORKSPACE", "./workspace")).resolve()
FIG_DIR = WORKSPACE / "figures"
RESULTS_DIR = WORKSPACE / "results" / "external_consolidated"
FIG_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"WORKSPACE: {WORKSPACE}")


In [ ]:
def load_json(path):
    p = Path(path)
    if not p.exists():
        return None
    return json.loads(p.read_text())

nb14 = load_json(WORKSPACE / "results" / "cptac_external_openclip" / "report.json")
nb15 = load_json(WORKSPACE / "results" / "cptac_external_osfm" / "report.json")
nb16 = load_json(WORKSPACE / "results" / "ptrc_platinum" / "report.json")
nb17 = load_json(WORKSPACE / "results" / "offtarget_mmr" / "report.json")

print(f"NB14 loaded: {nb14 is not None}")
print(f"NB15 loaded: {nb15 is not None}")
print(f"NB16 loaded: {nb16 is not None}")
print(f"NB17 loaded: {nb17 is not None}")


In [ ]:
rows = []

# 1. CPTAC-LUAD frozen OpenCLIP
if nb14:
    for r in nb14.get("results", []):
        if r.get("cohort") == "LUAD" and "AUROC" in r:
            rows.append({
                "label": "CPTAC-LUAD (frozen OpenCLIP B/16)",
                "category": "HRD on-target",
                "AUROC": r["AUROC"], "lo": r["AUROC_lo"], "hi": r["AUROC_hi"],
                "n": r["n"], "n_pos": r["n_pos"],
                "manuscript_AUROC": 0.671,
            })

# 2-4. CPTAC OSFM cohorts
if nb15:
    for r in nb15.get("results", []):
        coh = r.get("cohort")
        if coh in {"LUAD", "LUSC", "HNSCC"} and "AUROC" in r:
            rows.append({
                "label": f"CPTAC-{coh} (OSFM, within-cohort CV)",
                "category": "HRD on-target",
                "AUROC": r["AUROC"], "lo": r["AUROC_lo"], "hi": r["AUROC_hi"],
                "n": r["n"], "n_pos": r["n_pos"],
                "manuscript_AUROC": r.get("manuscript_AUROC"),
            })

# 5. PTRC platinum (single metric set per new NB16 schema)
if nb16:
    m = nb16.get("metrics", {})
    if m and "AUROC" in m:
        rows.append({
            "label": "PTRC-HGSOC (platinum resistance)",
            "category": "platinum response",
            "AUROC": m["AUROC"], "lo": m["AUROC_lo"], "hi": m["AUROC_hi"],
            "n": nb16.get("n_total"), "n_pos": nb16.get("n_resistant"),
            "manuscript_AUROC": 0.673,
        })

# 6. SurGen MMR (off-target)
if nb17:
    sg = nb17.get("surgen", {})
    if sg and "AUROC" in sg:
        rows.append({
            "label": "SurGen-CRC (MMR, off-target)",
            "category": "off-target",
            "AUROC": sg["AUROC"], "lo": sg["AUROC_lo"], "hi": sg["AUROC_hi"],
            "n": sg["n"], "n_pos": sg["n_pos"],
            "manuscript_AUROC": 0.674,
        })

# 7. TCGA-UCEC MMR (off-target)
if nb17:
    uc = nb17.get("tcga_ucec", {})
    if uc and "AUROC" in uc:
        rows.append({
            "label": "TCGA-UCEC (MMR, off-target)",
            "category": "off-target",
            "AUROC": uc["AUROC"], "lo": uc["AUROC_lo"], "hi": uc["AUROC_hi"],
            "n": uc["n"], "n_pos": uc["n_pos"],
            "manuscript_AUROC": 0.445,
        })

forest = pd.DataFrame(rows)
forest.to_csv(RESULTS_DIR / "external_summary.csv", index=False)
print(forest.to_string(index=False))


In [ ]:
if forest.empty:
    raise SystemExit("no external metrics available; run NB14-NB17 first")

cat_color = {"HRD on-target": "C0", "platinum response": "C2", "off-target": "C3"}
y_pos = np.arange(len(forest))[::-1]

fig, ax = plt.subplots(figsize=(7.5, 0.55 * len(forest) + 1.4))
for i, (_, r) in enumerate(forest.iterrows()):
    yy = y_pos[i]
    color = cat_color.get(r["category"], "0.4")
    err_lo = max(0.0, r["AUROC"] - r["lo"])
    err_hi = max(0.0, r["hi"] - r["AUROC"])
    ax.errorbar(r["AUROC"], yy, xerr=[[err_lo], [err_hi]],
                fmt="o", color=color, ecolor=color,
                markersize=8, capsize=3, lw=1.6)
    npos = r.get("n_pos")
    n = r.get("n")
    suffix = f"  (n={n}, +{npos})" if (n and npos is not None) else ""
    ax.text(1.02, yy, f"{r['AUROC']:.3f} [{r['lo']:.3f}-{r['hi']:.3f}]{suffix}",
            va="center", fontsize=8, transform=ax.get_yaxis_transform())

ax.axvline(0.5, color="0.5", lw=0.8, ls=":")
ax.set_yticks(y_pos)
ax.set_yticklabels(forest["label"].values, fontsize=9)
ax.set_xlabel("AUROC")
ax.set_xlim(0.0, 1.0)
ax.set_title("Figure 4A: external validation forest plot")

from matplotlib.lines import Line2D
handles = [Line2D([0], [0], marker="o", color="w", markerfacecolor=c, markersize=8, label=k)
           for k, c in cat_color.items() if k in forest["category"].values]
handles.append(Line2D([0], [0], color="0.5", lw=0.8, ls=":", label="random (0.5)"))
ax.legend(handles=handles, loc="lower left", fontsize=8, frameon=False)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig4a_forest_external.png", dpi=300, bbox_inches="tight")
fig.savefig(FIG_DIR / "fig4a_forest_external.pdf", bbox_inches="tight")
plt.show()


In [ ]:
table = forest.copy()
table["AUROC_with_CI"] = table.apply(
    lambda r: f"{r['AUROC']:.3f} ({r['lo']:.3f}-{r['hi']:.3f})", axis=1
)
table["delta_vs_manuscript"] = table.apply(
    lambda r: (r["AUROC"] - r["manuscript_AUROC"])
    if pd.notna(r.get("manuscript_AUROC")) else float("nan"),
    axis=1,
)
out_cols = ["label", "category", "n", "n_pos",
            "AUROC", "lo", "hi", "AUROC_with_CI",
            "manuscript_AUROC", "delta_vs_manuscript"]
table[out_cols].to_csv(RESULTS_DIR / "external_consolidated_table.csv", index=False)
print(table[out_cols].to_string(index=False))
print()
print(f"figures: {FIG_DIR / 'fig4a_forest_external.png'}")
print(f"summary: {RESULTS_DIR / 'external_consolidated_table.csv'}")
